## EDGAR Extraction Pipeline

**Goal:** Download 10-K annual reports for the companies selected in `01_explore_companies.ipynb` and extract the **Item 1A (Risk Factors)** section from each filing.

**Run cells top to bottom — no skipping.**

## 1. Imports & Configuration

In [1]:
import re
import time
import warnings
import requests
import pandas as pd
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning
from tqdm import tqdm
from pathlib import Path

warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

HEADERS = {
    "User-Agent": "daphne s_hsueh25@stud.hwr-berlin.de",
    "Accept-Encoding": "gzip, deflate",
}

DATA_DIR = Path("../data")
DATA_DIR.mkdir(exist_ok=True)
(DATA_DIR / "raw_filings").mkdir(exist_ok=True)
(DATA_DIR / "item1a").mkdir(exist_ok=True)

REQUEST_DELAY = 0.15

START_YEAR = 2010
END_YEAR   = 2026

# Set to a small number (e.g. 3) for a quick test run; None to process all
MAX_COMPANIES = None

print("Configuration ready.")
print(f"Saving data to: {DATA_DIR.resolve()}")

Configuration ready.
Saving data to: /Users/hdaphne/Desktop/bipm/NLP/topic-modeling-G5/data


## 2. Load Selected Companies

Loads `data/selected_companies.csv` produced by `01_explore_companies.ipynb`.

In [2]:
companies = pd.read_csv(DATA_DIR / "selected_companies.csv", dtype={"cik": str})
companies["cik"] = companies["cik"].str.zfill(10)

if MAX_COMPANIES:
    companies = companies.head(MAX_COMPANIES)

print(f"Loaded {len(companies)} companies")
print(f"\nBy sector:")
print(companies["sector"].value_counts().to_string())
companies.head()

Loaded 41 companies

By sector:
sector
Industrials               10
Consumer Discretionary     9
Real Estate                5
Financials                 4
Health Care                4
Communication Services     2
Consumer Staples           2
Information Technology     2
Energy                     1
Materials                  1
Utilities                  1


,ticker,company,sector,cik,total_10k,earliest_year,latest_year,error
0,LYV,Live Nation Entertainment,Communication Services,0001335258,14,2013.0,2026.0,NaN
1,SATS,EchoStar,Communication Services,0001415404,16,2011.0,2026.0,NaN
2,ULTA,Ulta Beauty,Consumer Discretionary,0001403568,17,2010.0,2026.0,NaN
3,LVS,Las Vegas Sands,Consumer Discretionary,0001300514,17,2010.0,2026.0,NaN
4,PHM,PulteGroup,Consumer Discretionary,0000822416,15,2012.0,2026.0,NaN


## 3. Helper Functions

In [3]:
SUBMISSIONS_API = "https://data.sec.gov/submissions/CIK{cik}.json"


def get_10k_filings(cik: str, start_year: int, end_year: int) -> pd.DataFrame:
    url = SUBMISSIONS_API.format(cik=cik)
    response = requests.get(url, headers=HEADERS)
    response.raise_for_status()
    data = response.json()
    time.sleep(REQUEST_DELAY)

    filings = data.get("filings", {}).get("recent", {})
    if not filings:
        return pd.DataFrame()

    df = pd.DataFrame({
        "form":             filings["form"],
        "filing_date":      filings["filingDate"],
        "accession_number": filings["accessionNumber"],
        "primary_document": filings["primaryDocument"],
    })

    df = df[df["form"] == "10-K"].copy()
    df["year"] = pd.to_datetime(df["filing_date"]).dt.year
    df = df[(df["year"] >= start_year) & (df["year"] <= end_year)]
    return df.reset_index(drop=True)

In [4]:
FILING_BASE_URL = "https://www.sec.gov/Archives/edgar/data/{cik}/{accession}/{document}"


def download_filing(cik: str, accession_number: str, primary_document: str) -> str:
    accession_clean = accession_number.replace("-", "")
    cik_numeric = str(int(cik))
    url = FILING_BASE_URL.format(
        cik=cik_numeric, accession=accession_clean, document=primary_document
    )
    response = requests.get(url, headers=HEADERS)
    response.raise_for_status()
    time.sleep(REQUEST_DELAY)
    return response.text

In [ ]:
ITEM_1A_PATTERN = re.compile(
    r'^[ \t]*item\s*1a[\s\W]{0,30}risk\s*factors[^\n]*\n(.+?)(?=^[ \t]*item\s*(?:1b|2)\b)',
    re.IGNORECASE | re.MULTILINE | re.DOTALL,
)
ITEM_1A_FALLBACK = re.compile(
    r'^[ \t]*item\s*1a[^\n]*\n(.{200,}?)(?=^[ \t]*item\s*(?:1b|2)\b)',
    re.IGNORECASE | re.MULTILINE | re.DOTALL,
)

# Matches a repeated Item 1A header inside already-captured content (TOC match case)
_NESTED_1A = re.compile(
    r'item\s*1a[\s\S]{0,50}?risk\s*factors[^\n]*\n([\s\S]+)',
    re.IGNORECASE,
)

# Only coordinating conjunctions are clear fragment starters in formal documents
_COORD_CONJ = re.compile(r'^(?:and|or|but)\b', re.IGNORECASE)


def _trim_leading_noise(text: str) -> str:
    """Remove header artifacts and TOC cross-references from the start of extracted text.

    Known failure modes (all stem from the regex matching a TOC entry):
    - Starts mid-word:   "sk Factors.\\nOur business..."
    - Starts with punct: '." This item...' or '," below and elsewhere...'
    - Starts mid-sent:   "and in the Financial Review section..."

    Steps:
    1. Re-anchor to a nested Item 1A header if one exists in the captured block.
    2. Strip a leading punctuation artifact when what immediately follows starts clean.
    3. If still a fragment (lowercase, punct, or coordinating conjunction), find the
       first proper sentence boundary inside the text.
    """
    # Step 1: re-anchor to nested Item 1A header
    nested = _NESTED_1A.search(text)
    if nested:
        candidate = nested.group(1).strip()
        if len(candidate) > 200:
            text = candidate

    # Step 2: strip a single leading punctuation/quote artifact
    stripped = text.lstrip()
    if stripped and stripped[0] in '.,;:!?)\'"–—':
        after_punct = re.sub(r'^[\W\s]+', '', stripped)
        if after_punct and after_punct[0].isupper():
            text = after_punct

    # Step 3: if still a fragment, find the first real sentence
    s = text.lstrip()
    is_fragment = (
        not s
        or not s[0].isupper()        # starts lowercase or punctuation
        or _COORD_CONJ.match(s)      # starts with and/or/but
    )
    if is_fragment:
        # Prefer a sentence boundary ('. Uppercase word >=50 chars of text')
        m = re.search(r'[.!?]["\'\s]+([A-Z]\w[^.!?\n]{50,})', text)
        if m:
            text = text[m.start(1):].strip()
        else:
            # Fall back to line scan
            for line in text.split('\n'):
                ls = line.strip()
                if ls and ls[0].isupper() and not _COORD_CONJ.match(ls) and len(ls) > 60:
                    text = text[text.index(line):].strip()
                    break

    return text


def extract_item_1a(html: str) -> str | None:
    soup = BeautifulSoup(html, "lxml")
    for tag in soup(["script", "style", "table"]):
        tag.decompose()
    # Preserve paragraph breaks; only collapse horizontal whitespace
    text = soup.get_text(separator="\n")
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text).strip()
    match = ITEM_1A_PATTERN.search(text) or ITEM_1A_FALLBACK.search(text)
    if not match:
        return None
    return _trim_leading_noise(match.group(1).strip())

## 4. Run Full Pipeline

For each company: fetch filing list → download each 10-K → extract Item 1A → save.

- `data/item1a/{cik}_{year}.txt` — one file per company per year
- `data/filing_index.csv` — master index with extraction status

In [ ]:
def run_pipeline(
    companies: pd.DataFrame,
    start_year: int,
    end_year: int,
    force: bool = False,
    force_ciks: set | None = None,
) -> pd.DataFrame:
    """Run the extraction pipeline.

    force       — re-extract every file (slow; use after regex changes affect all files)
    force_ciks  — re-extract only these CIKs (fast targeted fix)
    """
    records = []

    for _, company in tqdm(companies.iterrows(), total=len(companies), desc="Companies"):
        cik    = company["cik"]
        ticker = company["ticker"]
        name   = company["company"]

        try:
            filings = get_10k_filings(cik, start_year, end_year)
        except Exception as e:
            print(f"  Could not fetch filings for {ticker}: {e}")
            continue

        if filings.empty:
            print(f"  No 10-K filings found for {ticker} in range")
            continue

        for _, filing in filings.iterrows():
            year             = filing["year"]
            accession_number = filing["accession_number"]
            primary_document = filing["primary_document"]
            output_path      = DATA_DIR / "item1a" / f"{cik}_{year}.txt"

            should_force = force or (force_ciks is not None and cik in force_ciks)

            if output_path.exists() and not should_force:
                records.append({"ticker": ticker, "company": name, "cik": cik,
                                 "year": year, "accession": accession_number,
                                 "status": "skipped (already exists)", "path": str(output_path)})
                continue

            try:
                html = download_filing(cik, accession_number, primary_document)
            except Exception as e:
                records.append({"ticker": ticker, "company": name, "cik": cik,
                                 "year": year, "accession": accession_number,
                                 "status": f"download_failed: {e}", "path": None})
                continue

            item_1a = extract_item_1a(html)
            if item_1a:
                output_path.write_text(item_1a, encoding="utf-8")
                status = "success"
            else:
                status = "item1a_not_found"

            records.append({"ticker": ticker, "company": name, "cik": cik,
                             "year": year, "accession": accession_number,
                             "status": status, "path": str(output_path) if item_1a else None})

    return pd.DataFrame(records)


# CIKs with known start-of-text artifacts (punctuation, mid-sentence, mid-word)
# Abbott (1800), Applied Materials (6951), Marriott (66740), HCA (352915),
# Bio-Rad (84839), Cintas (723254), Simon Property (815097), BRO (79282),
# MDC Holdings (906163), Masco (945841)
_AFFECTED_CIKS = {
    "0000001800", "0000006951", "0000066740", "0000084839",
    "0000352915", "0000723254", "0000815097", "0000079282",
    "0000906163", "0000945841",
}

index = run_pipeline(companies, START_YEAR, END_YEAR, force_ciks=_AFFECTED_CIKS)
index.to_csv(DATA_DIR / "filing_index.csv", index=False)

print(f"\nDone. Saved index to {DATA_DIR / 'filing_index.csv'}")
print(index["status"].value_counts())

In [7]:
# Verify: check how many files still start mid-sentence
item1a_dir = DATA_DIR / "item1a"
mid_sentence = []
for path in sorted(item1a_dir.glob("*.txt")):
    first_char = path.read_text(encoding="utf-8")[:1]
    if first_char.islower():
        mid_sentence.append(path.name)

print(f"Files still starting mid-sentence: {len(mid_sentence)}")
for name in mid_sentence:
    snippet = (item1a_dir / name).read_text(encoding="utf-8")[:120].replace('\n', ' ')
    print(f"  {name}: {snippet}")

Files still starting mid-sentence: 4
  0000001800_2023.txt: and in the "Financial Review” section in Item 7. Management’s Discussion and Analysis of Financial Condition and Results
  0000001800_2024.txt: and in the "Financial Review” section in Item 7. Management’s Discussion and Analysis of Financial Condition and Results
  0000906163_2013.txt: of this Form 10-K. Further discussion of settlements, new orders and backlog activity by homebuilding reportable segment
  0000945841_2026.txt: isk Factors Cautionary Statement for Purposes of the “Safe Harbor” Provisions of the Private Securities Litigation Refor


## 5. Inspect Results & Build Corpus

In [8]:
print("=== Pipeline Summary ===")
print(f"Total filings processed : {len(index)}")
print(f"Successful extractions  : {(index['status'] == 'success').sum()}")
print(f"Item 1A not found       : {(index['status'] == 'item1a_not_found').sum()}")
print(f"Download failures       : {index['status'].str.startswith('download').sum()}")
print(f"\nYear range covered      : {index['year'].min()} - {index['year'].max()}")
print(f"Unique companies        : {index['ticker'].nunique()}")

successful = index[index["status"] == "success"]
print("\n=== Successful Extractions Per Year ===")
print(successful.groupby("year").size().sort_index().to_string())

=== Pipeline Summary ===
Total filings processed : 610
Successful extractions  : 60
Item 1A not found       : 118
Download failures       : 0

Year range covered      : 2010 - 2026
Unique companies        : 41

=== Successful Extractions Per Year ===
year
2010    1
2011    2
2012    1
2013    1
2014    2
2015    3
2016    4
2017    4
2018    4
2019    5
2020    5
2021    5
2022    5
2023    5
2024    5
2025    5
2026    3


In [9]:
def load_corpus(filing_index: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in filing_index[filing_index["status"] == "success"].iterrows():
        text = Path(row["path"]).read_text(encoding="utf-8")
        rows.append({"ticker": row["ticker"], "company": row["company"],
                     "cik": row["cik"], "year": row["year"], "text": text})
    return pd.DataFrame(rows)


corpus = load_corpus(index)
corpus.to_csv(DATA_DIR / "corpus.csv", index=False)

print(f"Corpus saved to {DATA_DIR / 'corpus.csv'}")
print(f"Shape: {corpus.shape}")
corpus.head()

Corpus saved to ../data/corpus.csv
Shape: (60, 5)


,ticker,company,cik,year,text
0,CCL,Carnival Corporation,0000815097,2026,You should carefully consider the following di...
1,CCL,Carnival Corporation,0000815097,2025,You should carefully consider the following di...
2,CCL,Carnival Corporation,0000815097,2024,You should carefully consider the following di...
3,CCL,Carnival Corporation,0000815097,2023,Forward-looking statements should not be relie...
4,CCL,Carnival Corporation,0000815097,2022,It is not possible to predict or identify all ...


In [10]:
# In the next notebook, load the corpus with:
# corpus = pd.read_csv("../data/corpus.csv")